# 2. Clusters, PCA, and Correlations

**Pipeline position:** runs after `1_data_prep.ipynb`. Reads `panel.parquet`
and produces three sets of artifacts.

## What this notebook does

1. **Pearson correlations** — computes the correlation between ECI and each
   feature in the category map (`CAT_MAP` in `_config.py`). Used by chart 4.
2. **PCA scatter** — projects the 1995 cross-section onto its first two
   principal components. Used by chart 26.
3. **PCA loadings** — saves PC1 and PC2 loadings for charts 27 and 28.
4. **Median ECI by cluster** — pre-aggregates ECI median + standard error
   over time, by cluster. Used by chart 6.

The k=5 cluster *assignments* themselves are not recomputed here; they come
from `clusters1995.csv` produced by the main pipeline (notebook 4) and are
already saved in `artifacts/clusters_1995.csv` by the previous notebook.

## Outputs

- `corr_with_eci.csv` — feature, short label, category, Pearson r
- `pca_scatter.csv` — country, cluster, PC1, PC2
- `pca_loadings.csv` — feature, short label, PC1 loading, PC2 loading,
  variance explained
- `median_eci_by_cluster.csv` — year, cluster, median ECI, standard error,
  count


In [1]:
import os, sys
sys.path.insert(0, '.')
import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

from _config import (ARTIFACTS, ECI_COL, CAT_MAP, NAME_MAP,
                     artifact_path)

panel = pd.read_parquet(artifact_path('panel.parquet'))
clusters_1995 = pd.read_csv(artifact_path('clusters_1995.csv'))

print(f'Panel: {panel["Country Code"].nunique()} countries, {len(panel):,} obs')


Panel: 49 countries, 1,225 obs


## Pearson correlations with ECI

For each feature in `CAT_MAP`, compute Pearson r against ECI on the panel.
Skip features that are missing from `Master.csv` or that have fewer than 20
non-null observations.


In [2]:
rows = []
for col, (short, cat) in CAT_MAP.items():
    if col not in panel.columns:
        continue
    sub = panel[[col, ECI_COL]].dropna()
    if len(sub) < 20:
        continue
    r = float(np.corrcoef(sub[col], sub[ECI_COL])[0, 1])
    rows.append(dict(feature=col, short=short, cat=cat, r=r))

corr_df = pd.DataFrame(rows).sort_values('r').reset_index(drop=True)
corr_df.to_csv(artifact_path('corr_with_eci.csv'), index=False)
print(f'  Saved corr_with_eci.csv ({len(corr_df)} features)')


  Saved corr_with_eci.csv (31 features)


## PCA on 1995 cross-section

Standardise the same numeric features the cluster pipeline uses, fit PCA on
the 1995 snapshot, save the per-country PC1/PC2 coordinates and the loadings.


In [3]:
# Numeric features used for the cluster space (mirrors NB4 in the main repo).
PCA_FEATS = [
    'Total_Production_Value_Per_Capita',
    'Total natural resources rents (% of GDP)',
    'Oil rents (% of GDP)',
    'Natural gas rents (% of GDP)',
    'Mineral rents (% of GDP)',
    'Forestry rents (% of GDP)',
    'GDP per capita (constant prices, PPP)',
    'Trade (% of GDP)',
    'Manufacturing',
    'Services',
]
pca_feats_present = [c for c in PCA_FEATS if c in panel.columns]

snap = panel[panel['Year'] == 1995].copy()
snap = snap.dropna(subset=pca_feats_present + ['ClusterLabels'])

X = StandardScaler().fit_transform(snap[pca_feats_present].values)
pca = PCA(n_components=2, random_state=42).fit(X)
PCs = pca.transform(X)

scatter_df = pd.DataFrame({
    'Country Code': snap['Country Code'].values,
    'Country':      snap['Country Name'].values,
    'ClusterLabels': snap['ClusterLabels'].values,
    'PC1':          PCs[:, 0],
    'PC2':          PCs[:, 1],
})
scatter_df.to_csv(artifact_path('pca_scatter.csv'), index=False)
print(f'  Saved pca_scatter.csv ({len(scatter_df)} countries)')
print(f'  PC1 = {pca.explained_variance_ratio_[0]*100:.1f}%, '
      f'PC2 = {pca.explained_variance_ratio_[1]*100:.1f}%, '
      f'total = {sum(pca.explained_variance_ratio_)*100:.1f}%')

loadings_df = pd.DataFrame({
    'feature': pca_feats_present,
    'short':   [NAME_MAP.get(c, c) for c in pca_feats_present],
    'PC1':     pca.components_[0],
    'PC2':     pca.components_[1],
    'PC1_var': pca.explained_variance_ratio_[0],
    'PC2_var': pca.explained_variance_ratio_[1],
})
loadings_df.to_csv(artifact_path('pca_loadings.csv'), index=False)
print(f'  Saved pca_loadings.csv ({len(loadings_df)} features)')


  Saved pca_scatter.csv (49 countries)
  PC1 = 28.6%, PC2 = 22.0%, total = 50.6%
  Saved pca_loadings.csv (10 features)


## Median ECI by cluster, over time

The chart needs median + standard error per (year, cluster). Pre-aggregate
here so the viz notebook does no GROUP BY.


In [4]:
traj = (
    panel.dropna(subset=['ClusterLabels', ECI_COL])
         .groupby(['Year', 'ClusterLabels'])[ECI_COL]
         .agg(['median', 'std', 'count'])
         .reset_index()
)
traj.columns = ['Year', 'ClusterLabels', 'median', 'std', 'n']
traj['se'] = traj['std'] / np.sqrt(traj['n'])
traj.to_csv(artifact_path('median_eci_by_cluster.csv'), index=False)
print(f'  Saved median_eci_by_cluster.csv ({len(traj)} rows)')


  Saved median_eci_by_cluster.csv (125 rows)


## Summary

| Artifact | Rows | Used by |
|---|---|---|
| corr_with_eci.csv | one per feature | chart 4 |
| pca_scatter.csv | one per country (1995) | chart 26 |
| pca_loadings.csv | one per feature | charts 27, 28 |
| median_eci_by_cluster.csv | one per (year, cluster) | chart 6 |

Run notebook 3 next.
